In [1]:
import chess
import random
import json
from tqdm import tqdm

OUTPUT_FILE = "synthetic_chess_dataset.jsonl"

In [12]:
def generate_templates(board, move, san) -> list[str]:
    piece = board.piece_at(move.from_square)
    target = chess.square_name(move.to_square)

    is_capture = board.is_capture(move)
    is_castle = board.is_castling(move)
    is_promotion = move.promotion is not None

    templates = []

    file_name = chess.FILE_NAMES[chess.square_file(move.from_square)]
    
    if is_castle:    # todo: check if this also works for black
        if chess.square_file(move.to_square) == 6:
            templates += [
                "Castle kingside",
                "Kingside castle",
                "Perform kingside castling",
                "King castles short",
                "Castle short",
                "Short castle", 
            ]
        else:
            templates += [
                "Castle queenside",
                "Queenside castle",
                "Perform queenside castling",
                "King castles long",
                "Castle long",
                "Long castle",
            ]
        return templates

    if is_promotion:
        promo_piece = chess.piece_name(move.promotion)
        templates += [
            f"Promote pawn to {promo_piece} on {target}",
            f"Advance pawn to {target} and promote to {promo_piece}",
            f"Move pawn to {target} and make it a {promo_piece}",
        ]
        return templates

    p_name = chess.piece_name(piece.piece_type)

    if is_capture:
        templates += [
            f"{p_name.capitalize()} captures on {target}",
            f"Capture the piece on {target} with the {p_name}",
            f"Take on {target} using the {p_name}",
            f"Use the {p_name} to capture on {target}",
        ]
    elif piece.piece_type == chess.PAWN:
        templates += [
            f"Advance pawn to {target}",
            f"Push the pawn to {target}",
            f"Move pawn to {target}",
            f"Advance {file_name} pawn",
        ]
    else:
        templates += [
            f"Move the {p_name} to {target}",
            f"Play {p_name} to {target}",
            f"Develop the {p_name} to {target}",
            f"{p_name.capitalize()} goes to {target}",
        ]

    templates.append(f"Play {san}")

    return templates

In [13]:
def generate_dataset(board):
    print(board)
    with open(OUTPUT_FILE, "w") as f:
        dataset = []
        legal_moves = list(board.legal_moves)
        for move in legal_moves:
            san = board.san(move)
            templates = generate_templates(board, move, san)
            for template in templates:
                dataset.append(
                    {
                        "san": san,
                        "desc": template
                    }
                )
            
            for data in dataset:
                f.write(json.dumps(data) + "\n")

In [14]:
if __name__ == "__main__":
    board = chess.Board("rnb1k1nr/p2p1ppp/5q2/1pbN1N1P/4PBP1/3P1Q2/PPP5/R4KR1 b kq - 4 17")
    generate_dataset(board)
    print("Dataset generation complete.")

r n b . k . n r
p . . p . p p p
. . . . . q . .
. p b N . N . P
. . . . P B P .
. . . P . Q . .
P P P . . . . .
R . . . . K R .
Dataset generation complete.
